# Electricity bill forecast from AI agent usage patterns (seasonal ARIMA)

This notebook trains a model that forecasts a user's **electricity bill** from two things it learns from their device:

- **How each AI agent is used:** Claude Code, GitHub Copilot, Ollama and so on, each with its own energy pattern.
- **When it is used:** each agent's weekly routine, hour by hour, which also sets the price of each hour on a Peak/Off-Peak tariff.

The result of fine-tuning is **one seasonal ARIMA per AI agent**. Their hourly forecasts add up to the AI part of the bill.

| Step | What it does | Data |
|---|---|---|
| 1. Philippine grid data | Builds hourly Luzon demand from IEMOP's public 5-minute market data | IEMOP RTD Regional Summaries |
| 2. Pre-train | Tries several seasonal ARIMA structures on Luzon and ranks them on held-out days | Luzon hourly MWh |
| 3. Device readings | Turns the collector's 2-second readings into hourly energy **per AI agent** | this project's database |
| 4. Usage patterns | Usage levels (idle / light / moderate / heavy), and when and how heavily each agent is used | device hourly Wh |
| 5. Fine-tune | One ARIMA per agent, built on that agent's weekly routine and started from the Luzon coefficients; checked on held-out days | per-agent hourly Wh |
| 6. Forecast the bill | Baseline bill + AI cost so far this cycle + the forecast for the hours left, priced by hour | models + `backend/.env` |

**What "pre-train" and "fine-tune" mean here.** ARIMA has a handful of parameters and is always fitted to the series it forecasts. Luzon's grid data, which has a long, clean daily and weekly rhythm, ranks the model *structures* and supplies *starting* coefficients. Each agent's own readings set the final ones. Everything is scored on days held out of training. If a simple baseline ("same hour yesterday") forecasts as well as the models, the notebook says so.

**Configuration.** Every training setting is in `config.json` in this folder: how many days of Luzon data, the candidate structures, the held-out days, how gaps in the readings are treated, and so on. The next cell prints the settings in effect. To change one, edit `config.json` and restart the kernel. `wattcast/config.py` explains each setting.

**Iterations, not epochs.** ARIMA training has no epochs. A neural network passes over the data many times; an ARIMA is fitted in one optimization, where statsmodels adjusts a handful of coefficients by maximum likelihood until the log-likelihood stops improving. Every iteration of that optimizer uses the whole series, so its iterations are the nearest thing to epochs. `model.maxiter` in `config.json` is the most a fit may take (200). The training logs and tables show how many each fit took and whether it converged. Set `SHOW_ITERATIONS = True` below to print the log-likelihood after every iteration.

**Checkpoints.** Training fits many models one after another. Each one is saved the moment it finishes, in a `checkpoints` folder next to the final model, together with the configuration of the run. If a run is interrupted, running the cell again picks up where it stopped. A checkpoint is reused only when the data, the model structure and the settings are exactly the same.

Use the kernel from `training/arima_forecast/.venv` (see the README). The code lives in the `wattcast/` package, which the numbered scripts also use.

In [3]:
import json
import os
from contextlib import closing

import pandas as pd
from IPython.display import Markdown, display

from wattcast import iemop, model, patterns, pipeline, plots, readings, settings
from wattcast.config import CONFIG
from app.services import storage

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

DOWNLOAD = True     # False: use only the IEMOP files already in data/raw/iemop
QUICK = False       # True: try only 2 model structures on Luzon (faster)
FRESH = False       # True: ignore saved checkpoints and fit everything again
SHOW_ITERATIONS = False  # True: print the log-likelihood after every optimizer iteration of every fit
DEVICE_ID = None    # None: the device with the most readings

CONFIG["model"]["show_iterations"] = SHOW_ITERATIONS
print(json.dumps(CONFIG, indent=2))


def warn_not_converged(run):
    if run["not_converged"]:
        display(Markdown(f"**{len(run['not_converged'])} fit(s) stopped at the iteration cap without converging:** "
                         + "; ".join(run["not_converged"])
                         + ". Raise `model.maxiter` in `config.json`, restart the kernel and run again."))

{
  "timezone": "Asia/Manila",
  "step_minutes": 15,
  "checkpoints": true,
  "luzon": {
    "days": 30,
    "folds": 7,
    "horizon_hours": 24,
    "min_train_hours": 48,
    "candidates": [
      [
        [
          1,
          0,
          1
        ],
        [
          0,
          0,
          0,
          0
        ]
      ],
      [
        [
          2,
          0,
          1
        ],
        [
          0,
          0,
          0,
          0
        ]
      ],
      [
        [
          2,
          0,
          0
        ],
        [
          0,
          0,
          0,
          0
        ]
      ],
      [
        [
          1,
          0,
          1
        ],
        [
          1,
          0,
          1,
          "day"
        ]
      ],
      [
        [
          2,
          0,
          0
        ],
        [
          1,
          0,
          0,
          "day"
        ]
      ],
      [
        [
          0,
          1,
          1
        

## 1. Philippine grid data (IEMOP)

IEMOP, which runs the Wholesale Electricity Spot Market, publishes **RTD Regional Summaries**: one public file a day (no login), one row per 5-minute interval, region and commodity. The public page keeps about the last 90 days. Saved files are kept, so re-running this every few weeks builds a longer history.

The Luzon demand is the `MKT_REQT` column (MW) on rows with `REGION_NAME = CLUZ` and `COMMODITY_TYPE = En`. An hour's demand is the average of its twelve intervals, which is also its MWh. An hour with fewer than 10 intervals is left missing.

In [4]:
if DOWNLOAD:
    new = iemop.download(settings.RAW_IEMOP)
    print(f"Downloaded {len(new)} new daily files into {settings.RAW_IEMOP}")
luzon = iemop.hourly_demand(settings.RAW_IEMOP)
os.makedirs(settings.PROCESSED, exist_ok=True)
luzon.to_csv(settings.LUZON_HOURLY, index_label="hour")
print(f"Luzon: {luzon.notna().sum()} hours, {luzon.index[0]} to {luzon.index[-1]} ({luzon.isna().sum()} missing)")
luzon.tail()

KeyboardInterrupt: 

In [ ]:
grid = patterns.study_grid(luzon)
os.makedirs(settings.LUZON_DIR, exist_ok=True)
with open(os.path.join(settings.LUZON_DIR, "luzon_patterns.md"), "w") as f:
    f.write(patterns.grid_markdown(grid))
s = grid["seasonality"]
print(f"Average {grid['avg_mw']:,.0f} MW, peak {grid['peak_mw']:,.0f} MW at {grid['peak_at']}")
print(f"Correlation with the same hour a day earlier: {s['daily']}; a week earlier: {s['weekly']}")
plots.weekday_weekend(grid["profile"], "Luzon demand by hour of day", "MW");

## 2. Pre-train on Luzon

Every series, Luzon or one AI agent on a laptop, goes through the same transform before fitting:

$$z = \frac{\log(1 + y) - \text{mean}}{\text{std}}$$

The log makes a heavy hour and a light hour differ by a ratio, as usage levels do, and keeps forecasts from going below zero. Standardizing puts the grid (thousands of MWh) and one laptop (a few Wh) on the same scale, so coefficients learned on one are a sensible start for the other.

The model is

$$z_t = \text{routine}_t + c \cdot \text{holiday}_t + \text{SARIMA}(p,d,q)(P,D,Q)_{24}\ \text{errors}$$

- **routine** is the series' own weekly pattern: its typical level in each of the 168 hours of the week, learned from the training hours. An hour with few readings is pulled toward the average for that hour on weekdays or weekends. How hard depends on how regular the series is: a habit that repeats every week is kept, and a one-off is smoothed away.
- **holiday** marks Philippine public holidays.
- The **seasonal ARIMA** part, with a 24-hour season, is fitted to the departures from the routine: how a day runs above or below the usual, and how that carries into the next hours. The routine enters as it is, with no fitted multiplier.

For each candidate structure, the backtest fits it on the hours before the held-out days (`luzon.folds` in `config.json`), then forecasts each of those days 24 hours ahead using only earlier hours. The coefficients stay as fitted, and the routine is re-learned from the hours before each held-out day.

The main score is **MASE**: the average error divided by the training error of "same hour yesterday". Below 1 beats that rule. **day_total_error** is how far off each held-out day's total was, which is the closest of these scores to "how far off is the bill".

The log line for each structure ends with the optimizer iterations its fit took. The table below repeats that in the `iterations` and `converged` columns.

This step takes a few minutes the first time. Each structure is checkpointed in `artifacts/luzon/checkpoints` as it finishes, so a second run, or a run resumed after an interruption, loads those and takes seconds.

In [ ]:
y_luzon = luzon.iloc[-CONFIG["luzon"]["days"] * 24:]
luzon_result = pipeline.pretrain(y_luzon, model.CANDIDATES[:2] if QUICK else model.CANDIDATES,
                                 ckpt=pipeline.luzon_checkpoints(fresh=FRESH))
with open(settings.LUZON_MODEL, "w") as f:
    json.dump(luzon_result, f, indent=2)
run = luzon_result["run"]
print(f"{run['checkpoints']['fitted']} fitted, {run['checkpoints']['from_checkpoint']} from checkpoints, {run['seconds']:.0f}s")
print(f"Model: {settings.LUZON_MODEL}")
print(f"Checkpoints and run configuration: {settings.LUZON_CHECKPOINTS}")
warn_not_converged(run)

In [ ]:
table = pd.DataFrame([{"model": r["spec"], **r["metrics"], "aic": r["aic"], "iterations": r["iterations"],
                       "converged": r["converged"]} for r in luzon_result["ranking"]]
                     + [{"model": f"baseline: {k}", **v} for k, v in luzon_result["baselines"].items()])
display(table.set_index("model"))
plots.backtest_bars([(r["spec"], r["metrics"]["mase"], False) for r in luzon_result["ranking"]]
                    + [(f"baseline: {k}", v["mase"], True) for k, v in luzon_result["baselines"].items()],
                    "Luzon: held-out error (MASE, lower is better)");

## 3. Device readings, per AI agent

These are the readings the device reader (dashboard, or `backend/collect.py`) stores every 2 seconds, from the database set in `backend/.env`. For each clock hour (Philippine time):

- `measured_h` is the share of the hour the reader ran.
- `ai_wh` is the AI energy measured: watts × seconds of every AI app row, in Wh. It's all power drawn on this device, so all of it is on the bill.
- `wh` is what the models fit: `ai_wh`, or missing when the hour is unknown.

**An agent** is the app that drew the power (Claude Code, GitHub Copilot, Ollama, …). Its models and the commands it runs ("tool runs") all count as that agent. An agent gets its own model once it has 24 hours of use and 2% of the AI energy. Smaller ones are forecast together as "Other AI apps".

**Hours the reader didn't run** (`device.gaps` in `config.json`):

| `device.gaps` | Rule |
|---|---|
| `"day"` (default) | On a day the reader ran, hours without readings count as no AI use (computer off or asleep), as in the backend's daily totals. A day with no readings is unknown. |
| `"missing"` | Only hours the reader ran at least half of count, scaled to the full hour. Everything else is unknown. |
| `"zero"` | Every hour without readings counts as no AI use. |

In [ ]:
cfg = settings.backend_config()
print("Database:", storage.describe(cfg.DATABASE))
with closing(storage.connect(cfg.DATABASE)) as conn:
    devices = pd.DataFrame(storage.list_devices(conn))
    if devices.empty:
        raise RuntimeError("No devices with readings yet: start the device reader, then re-run from here.")
    devices["last_seen"] = pd.to_datetime(devices["last_seen"], unit="s", utc=True).dt.tz_convert(settings.TZ)
    display(devices[["id", "name", "os", "model", "samples", "last_seen"]])
    if DEVICE_ID is None:
        DEVICE_ID = int(devices.sort_values("samples", ascending=False).iloc[0]["id"])
    hourly, by_app = readings.hourly_ai_energy(conn, DEVICE_ID)

hourly.to_csv(settings.device_data(DEVICE_ID, "hourly.csv"))
by_app.to_csv(settings.device_data(DEVICE_ID, "by_app.csv"), index=False)
agents = readings.agent_series(hourly, by_app)
y = hourly["wh"]
print(f"Device {DEVICE_ID}: {y.notna().sum()} known hours on {len(set(y.dropna().index.date))} days, "
      f"{hourly['ai_wh'].sum() / 1000:.4f} kWh of AI, {y.index[0]} to {y.index[-1]}")
print("Agents:", ", ".join(agents.columns))
agents.dropna().tail()

## 4. Usage patterns: how heavily, which agent, and when

Each known hour gets a **usage level** from its average AI power:

| Level | Average AI power in the hour |
|---|---|
| idle | below 1 W: AI apps open, not working |
| light | the lowest third of this device's active hours |
| moderate | the middle third |
| heavy | the top third |

The cut points are the device's own percentiles, so "heavy" means heavy for that machine, whether it's a laptop running Claude Code or a desktop running a 70B model.

The second table is **per agent**: its share of the AI energy, how many hours it's in use, how heavily, its busiest hours, and how much of its energy falls in Meralco peak hours. The charts show when each agent is used. That weekly routine is what each agent's model is built on in step 5.

In [ ]:
report = patterns.study_device(hourly, by_app, agents)
with open(settings.device_artifact(DEVICE_ID, "patterns.json"), "w") as f:
    json.dump(report, f, indent=2)
with open(settings.device_artifact(DEVICE_ID, "patterns.md"), "w") as f:
    f.write(patterns.device_markdown(DEVICE_ID, report))

th, s = report["thresholds_w"], report["seasonality"]
print(f"Levels: idle < {th['active_w']} W <= light <= {th['light_max_w']} W < moderate <= {th['moderate_max_w']} W < heavy")
print(f"Share of AI energy in Meralco peak hours: {report['energy_in_meralco_peak_hours']}")
print(f"Correlation with the same hour a day earlier: {s['daily']}; a week earlier: {s['weekly']}")
levels = pd.DataFrame(report["levels"])
levels["top_agents"] = [", ".join(a["name"] for a in l["top_agents"]) for l in report["levels"]]
display(levels.set_index("level"))
display(pd.DataFrame(report["agents"]).drop(columns="profile").set_index("agent"))

In [ ]:
plots.usage_heatmap(patterns.level_of(y, th), f"Device {DEVICE_ID}: AI usage level by hour")
plots.agent_profiles(report["agents"]);

## 5. Fine-tune: one ARIMA per AI agent

For each agent, these candidates are fitted on its hours before the held-out days:

- **Routine only:** the agent's weekly routine with no ARIMA terms. This is what ARIMA has to improve on.
- **The best Luzon structures, warm:** the optimizer starts from the Luzon coefficients of the same structure.
- **The same structures, cold:** it starts from statsmodels' defaults.

The candidate with the lowest held-out error is refitted on all of the agent's hours. For an agent whose use is fully explained by its weekly routine, that can be "Routine only". Set `device.allow_routine_only` to `false` in `config.json` to always choose an ARIMA; the routine alone is then scored for comparison only. Then the agents' held-out forecasts are **added up** and compared with the baselines on the device's total AI energy. That total is what the bill depends on.

Every fit is checkpointed in `artifacts/devices/device_<id>_checkpoints`, with the run's configuration and the bill settings from `backend/.env`. Fine-tuning needs at least 72 known hours on 3 or more days (`device.min_hours`, `device.min_days`). With fewer, this cell says how much more to collect, and the cells after it are skipped.

In [ ]:
try:
    saved = pipeline.finetune(agents, DEVICE_ID, pipeline.load_luzon(), bill_cfg=cfg,
                              ckpt=pipeline.device_checkpoints(DEVICE_ID, fresh=FRESH))
except ValueError as e:
    saved = None
    display(Markdown(f"**Device {DEVICE_ID} can't be fine-tuned yet:** {e}"))
if saved:
    with open(settings.device_artifact(DEVICE_ID, "model.json"), "w") as f:
        json.dump(saved, f, indent=2)
    run = saved["run"]
    print(f"{run['checkpoints']['fitted']} fitted, {run['checkpoints']['from_checkpoint']} from checkpoints, {run['seconds']:.0f}s")
    print(f"Model: {settings.device_artifact(DEVICE_ID, 'model.json')}")
    print(f"Checkpoints and run configuration: {run['checkpoints']['folder']}")
    warn_not_converged(run)

In [ ]:
if saved:
    tried = pd.DataFrame([{"agent": name, "model": t["spec"], "start": t["start"], "chosen": i == 0, **t["metrics"],
                           "iterations": t["iterations"], "converged": t["converged"]}
                          for name, a in saved["agents"].items() for i, t in enumerate(a["tried"])])
    display(tried.set_index(["agent", "model", "start"]))
    display(pd.DataFrame(saved["total"]).T)
    plots.backtest_bars([(("ARIMA per agent, summed" if k == "arima_by_agent" else f"baseline: {k}"), v["mase"],
                          k != "arima_by_agent") for k, v in saved["total"].items()],
                        f"Device {DEVICE_ID}: held-out error on total AI energy (MASE, lower is better)")
    if not saved["beats_baselines"]:
        display(Markdown("**Note:** a simple baseline forecast the held-out days at least as well as the per-agent "
                         "models. Collect more days of readings before relying on this forecast."))

## 6. Forecast the electricity bill

$$\text{projected bill} = \text{baseline bill} + \text{AI cost measured so far this cycle} + \text{AI cost forecast for the hours left}$$

- The **baseline bill** (the bill without AI) and the billing cycle come from `backend/.env`.
- Each agent's model forecasts its hourly energy from the current hour to the end of the cycle. If the reader stopped a while ago, the models bridge the gap.
- Each hour is priced on the tariff: a flat `ELECTRICITY_RATE`, or Meralco Peak/Off-Peak (`TARIFF=pop`) using the backend's own schedule. On Peak/Off-Peak, *when* an agent is used changes what it costs.
- Ranges are 80% (10th to 90th percentile) from 500 simulated futures per agent, so totals over many hours have their own range instead of a sum of hourly ranges.
- Simulated futures only vary hour to hour around the learned routine. They don't know the routine itself may be off. So a total's range is never narrower than the error the backtest found on held-out day totals in step 5.

In [ ]:
if saved:
    bill_hourly, summary = pipeline.forecast(saved, agents[list(saved["agents"])], hourly, cfg)
    bill_hourly.round(5).to_csv(settings.device_artifact(DEVICE_ID, "forecast.csv"))
    with open(settings.device_artifact(DEVICE_ID, "forecast.json"), "w") as f:
        json.dump(summary, f, indent=2)

    b, tariff = summary["bill"], summary["tariff"]
    print(f"Billing cycle {b['cycle']['start']} to {b['cycle']['end']}, tariff: {tariff['tariff']} (P{tariff['rate']}/kWh"
          + (f"; peak P{tariff['peak_rate']}, off-peak P{tariff['offpeak_rate']})" if tariff["tariff"] == "pop" else ")"))
    print(f"Projected bill:      P{b['projected_bill']:,.2f}  (P{b['projected_bill_low']:,.2f} to P{b['projected_bill_high']:,.2f})")
    print(f"  baseline           P{b['baseline_bill']:,.2f}")
    print(f"  AI so far          P{b['ai_cost_so_far']:,.2f}  ({b['hours_measured_so_far']} of {b['hours_elapsed']} hours measured)")
    print(f"  AI still to come   P{b['ai_cost_remaining']:,.2f}  (P{b['ai_cost_remaining_low']:,.2f} to P{b['ai_cost_remaining_high']:,.2f})")
    print(f"  AI share of bill   {b['ai_share_of_bill']:.2%}")
    print(f"Ranges are at least +/- {summary['backtest_day_total_error']:.0%}, the backtest's error on held-out day totals.")
    display(pd.DataFrame(summary["totals"]).T[["hours", "kwh", "cost", "cost_low", "cost_high"]])
    plots.cost_by_agent(summary["cost_by_agent_rest_of_cycle"])
    plots.forecast_cost(bill_hourly, 72, f"Device {DEVICE_ID}: AI cost per hour, next 72 hours (PHP)")
else:
    print("No fine-tuned model yet: collect more readings, then run step 5 again.")

In [ ]:
bill_hourly.head(24).round(4) if saved else None

## Files written

| File | What |
|---|---|
| `data/processed/luzon_hourly.csv` | Luzon hourly demand (MWh) |
| `artifacts/luzon/luzon_pretrained.json` | Ranked structures with their Luzon coefficients, the comparison, and the run record (configuration, versions, timing) |
| `artifacts/luzon/checkpoints/` | One file per fitted structure, plus `config.json` for the run |
| `data/processed/device_<id>_hourly.csv`, `_by_app.csv` | The device's hourly AI energy, in total and per agent and model |
| `artifacts/devices/device_<id>_patterns.md` / `.json` | Usage levels, and when and how heavily each agent is used |
| `artifacts/devices/device_<id>_model.json` | One fine-tuned model per agent, the backtest, level cut points, and the run record |
| `artifacts/devices/device_<id>_checkpoints/` | One file per fitted candidate per agent, plus `config.json` for the run |
| `artifacts/devices/device_<id>_forecast.csv` / `.json` | Hourly forecast per agent, totals, and the projected bill |

Device files describe one person's computer use and are git-ignored. The Luzon files are public data.

**To use it in the app:** the backend can load `device_<id>_model.json` and call `wattcast.pipeline.forecast` on the latest hourly readings. That would give `/api/forecast` the projected bill with its range, the hourly series, and the cost per agent.